# HypeCheck: do headline-style posts predict memecoin direction?
**Veritas AI International High School Hackathon 2026 · AI + Finance**

**Team:** Nathaniel Wong (model), Aaden Yick (data), Alastair Chu (demo and write-up) · Malvern College Hong Kong.

**Pitch:** HypeCheck helps a beginner crypto investor test whether bullish social posts deserve forecasting trust, by comparing sentiment-based next-day forecasts against simple rules and actual market outcomes.

**How to run:** Runtime → Run all. No keys, uploads or live Binance API calls. The default uses verified cached outputs from a pinned pretrained sentiment model and trains direction classifiers afresh. A recompute switch reruns frozen sentiment inference. The encoder is not fine-tuned.

This expands the original study to **13 memecoins**. There is no 40-tweet test and no 15/25 sentiment quota: all eligible naturally occurring posts are retained. X posts are our headline-style source, not newspaper headlines or a complete X feed.

The section sequence follows the sample: load → inspect → split → baseline → first model → mistakes → later-data test → measured improvement → demo → results → limitations and sources. New experiment, same archive: this is exploratory, not independent confirmation.

*This project is a prototype and is not financial advice.*

In [1]:
import sys,subprocess,os,json,hashlib,urllib.request,base64,zlib
from pathlib import Path
IN_COLAB='google.colab' in sys.modules
if IN_COLAB:
    pins=['numpy==2.5.4','pandas==3.1.1','pyarrow==26.0.1','scikit-learn==1.9.1','requests==2.34.2','transformers==5.23.0','safetensors==0.8.0']
    subprocess.run([sys.executable,'-m','pip','install','-q',*pins],check=True)
for name in ['data','docs','artifacts']:Path(name).mkdir(exist_ok=True)
print('Runtime:',sys.version,'Colab:',IN_COLAB)
print('The default direction classifiers are trained during this run.')

Runtime: 3.12.14 (main, Sep 24 2026, 17:57:43) [Clang 22.1.3 ] Colab: False
The default direction classifiers are trained during this run.


## 1. Load the data
The original posts are downloaded from a fixed Hugging Face revision and SHA-256 checked. The daily market panel contains actual Binance spot OHLCV bars for DOGE, SHIB, PEPE, BONK, FLOKI, WIF, TRUMP, NEIRO, PENGU, PNUT, TURBO, BOME and MEME, ending before the fixed evaluation horizon.

For reliable Colab access, the notebook embeds compressed, hash-checked copies of the exact price panel and frozen pretrained sentiment cache. These are data, not cached forecasts. We cite Binance's market-data terms, which remain separate from our code licence. Our exact source code is embedded too; no moving-branch dependency or credentials.

In [2]:
SOURCES = {'pipeline.py': '"""HypeCheck: reproducible retrospective experiment, never a trading service."""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport time\nfrom datetime import datetime, timezone\n\nimport numpy as np\nimport pandas as pd\nimport requests\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import accuracy_score, balanced_accuracy_score, brier_score_loss\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import OneHotEncoder, StandardScaler\n\nROOT = Path(__file__).resolve().parent\nDATASET = "StephanAkkerman/financial-tweets-crypto"\nDATASET_REV = "c4b17a94a667af3da1c9ed5144f998e3a4f29966"\nDATASET_SHA256 = "81b2f7563a44c1a103e9468a83d52319dfeae0fc616422b4a4df90d8d684b960"\nMODEL = "StephanAkkerman/FinTwitBERT-sentiment"\nMODEL_REV = "da059da3b3bbcb43f9ed1aeb5ae61644010c7e1e"\n# Declared secondary inputs, from the independent HypeCheck data package.\n# Both files are pinned by SHA-256 so a swapped or edited copy fails closed.\nHOURLY_FILE = "market-hourly.csv"\nHOURLY_SHA256 = "d6819bf4ff46c31c88d62130904e858deb0191125b2603910755feeba8cce003"\nEXPANDED_FILE = "expanded-daily.csv"\nEXPANDED_SHA256 = "2aab9bbe2c0d6782e04ec7902e3414ad564bc09eb35dccb802c75d45b2324935"\nINDEPENDENT_FILE = "independent-post-scores.csv"\nINDEPENDENT_SHA256 = "bb35778b88edeb9e0b11bc2ac9fba6068bf472614a48445eff447c79965e17a2"\nINDEPENDENT_MODEL_REV = "c079da7618e551d182ffdf68ef7aa25cb9a79667"\nCOINS = ("DOGE", "SHIB", "PEPE", "BONK", "FLOKI")\nSTART = pd.Timestamp("2024-03-01", tz="UTC")  # after this checkpoint was public\nEND = pd.Timestamp("2026-10-22", tz="UTC")  # exclusive, fixed historical horizon\nMARKET = ["ret1", "ret3", "ret7", "vol7", "range1", "volume_change"]\nATTENTION = ["post_count", "has_posts"]\nSOCIAL = ATTENTION + ["sentiment", "disagreement", "bullish", "bearish"]\n# Every extended value is computed only from bars that closed BEFORE cutoff t.\nBTC = ["btc_ret1", "btc_ret7"]\nINTRADAY = ["hourly_vol1", "hourly_range1", "last_hour_ret", "taker_ratio1"]\nMARKET_EXT = MARKET + BTC + INTRADAY\n\n\ndef atomic_json(path: Path, value):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    payload = json.dumps(value, ensure_ascii=False, allow_nan=False, indent=2)\n    tmp = path.with_suffix(path.suffix + ".part")\n    tmp.write_text(payload, encoding="utf-8")\n    tmp.replace(path)\n\n\ndef digest(path: Path):\n    with path.open("rb") as f:\n        return hashlib.file_digest(f, "sha256").hexdigest()\n\n\ndef request(url, **kwargs):\n    """Bounded public GETs only; never accept an input URL from the web UI."""\n    for attempt in range(3):\n        r = requests.get(url, timeout=(10, 60), **kwargs)\n        if r.status_code == 429 or r.status_code >= 500:\n            if attempt < 2:\n                time.sleep(2 ** attempt)\n                continue\n        r.raise_for_status()\n        return r\n    raise RuntimeError("Public data request failed")\n\n\ndef load_posts():\n    path = ROOT / "data" / "posts.parquet"\n    path.parent.mkdir(parents=True, exist_ok=True)\n    if not path.exists():\n        url = f"https://huggingface.co/datasets/{DATASET}/resolve/{DATASET_REV}/data/train.parquet"\n        r = request(url)\n        if len(r.content) > 50_000_000 or not r.content.startswith(b"PAR1"):\n            raise ValueError("Unexpected dataset payload; refusing to parse")\n        temp = path.with_suffix(".part")\n        temp.write_bytes(r.content)\n        temp.replace(path)\n    if digest(path) != DATASET_SHA256:\n        raise ValueError("Dataset SHA-256 does not match pinned Hugging Face LFS object")\n    raw = pd.read_parquet(path)\n    required = {"tweet_id", "created_at", "text", "tickers", "tweet_type", "first_posted_at"}\n    if not required <= set(raw):\n        raise ValueError(f"Missing columns: {required - set(raw)}")\n    if raw.tweet_id.isna().any() or raw.tweet_id.duplicated().any():\n        raise ValueError("Source has missing or duplicate tweet IDs")\n    raw["created_at"] = pd.to_datetime(raw.created_at, utc=True, errors="coerce")\n    raw["first_posted_at"] = pd.to_datetime(raw.first_posted_at, utc=True, errors="coerce")\n    raw["available_at"] = raw[["created_at", "first_posted_at"]].max(axis=1)\n    # A source collection time is conservative availability, not proof of live delivery.\n    valid = raw.created_at.notna() & raw.available_at.notna() & raw.text.notna()\n    original = raw[valid & raw.tweet_type.eq("tweet")].copy()\n    original["text"] = original.text.astype(str).str.strip()\n    original = original[original.text.str.len().between(8, 4000)]\n    original = original.sort_values("available_at").drop_duplicates("text", keep="first")\n    original = original[(original.available_at >= START - pd.Timedelta(days=1)) & (original.available_at < END)]\n    original["tickers"] = original.tickers.apply(lambda x: list(x) if isinstance(x, (list, np.ndarray)) else [])\n    original = original[original.tickers.apply(lambda xs: bool(set(xs) & set(COINS)))].copy()\n    original["date"] = original.available_at.dt.floor("D") + pd.Timedelta(days=1)\n    return raw, original\n\n\ndef load_prices(coin):\n    if coin not in COINS:\n        raise ValueError("Unknown coin")\n    path = ROOT / "data" / f"{coin}-prices.csv"\n    if path.exists():\n        prices = pd.read_csv(path)\n        prices["date"] = pd.to_datetime(prices.date, utc=True)\n    else:\n        rows = []\n        cursor = int(pd.Timestamp("2024-01-01", tz="UTC").timestamp() * 1000)\n        stop = int(END.timestamp() * 1000)\n        for _ in range(10):\n            batch = request("https://api.binance.com/api/v3/klines", params={\n                "symbol": coin + "USDT", "interval": "1d", "startTime": cursor,\n                "endTime": stop - 1, "timeZone": "0", "limit": 1000,\n            }).json()\n            if not isinstance(batch, list):\n                raise ValueError("Invalid price response")\n            if not batch:\n                break\n            rows.extend(batch)\n            next_cursor = int(batch[-1][0]) + 86_400_000\n            if next_cursor <= cursor:\n                raise ValueError("Price pagination did not advance")\n            cursor = next_cursor\n            if cursor >= stop:\n                break\n            time.sleep(0.15)\n        else:\n            raise RuntimeError("Price pagination exceeded bounded range")\n        if not rows:\n            raise ValueError(f"No prices for {coin}")\n        prices = pd.DataFrame({\n            "date": pd.to_datetime([r[0] for r in rows], unit="ms", utc=True),\n            "open": [float(r[1]) for r in rows], "high": [float(r[2]) for r in rows],\n            "low": [float(r[3]) for r in rows], "close": [float(r[4]) for r in rows],\n            "volume": [float(r[7]) for r in rows],\n        })\n        temp = path.with_suffix(".part")\n        prices.to_csv(temp, index=False)\n        temp.replace(path)\n    prices = prices.sort_values("date").reset_index(drop=True)\n    if prices.date.duplicated().any() or prices.empty:\n        raise ValueError("Invalid price date index")\n    if not prices.date.eq(prices.date.dt.floor("D")).all() or prices.date.max() >= END:\n        raise ValueError("Price bars must open at UTC midnight within the fixed horizon")\n    if not (prices.date.diff().dropna() == pd.Timedelta(days=1)).all():\n        raise ValueError(f"Missing daily prices for {coin}; no silent forward filling")\n    vals = prices[["open", "high", "low", "close", "volume"]].to_numpy(float)\n    if not np.isfinite(vals).all() or (vals[:, :4] <= 0).any() or (vals[:, 4] < 0).any():\n        raise ValueError("Non-finite or invalid market data")\n    if (prices.high < prices[["open", "close", "low"]].max(axis=1)).any() or (prices.low > prices[["open", "close", "high"]].min(axis=1)).any():\n        raise ValueError("Invalid OHLC bounds")\n    return prices\n\n\ndef validate_sentiment_scores(values):\n    """Treat the local JSON cache as untrusted numeric data, not model authority."""\n    if not isinstance(values, dict) or set(values) != {"bullish", "bearish", "neutral"}:\n        raise ValueError("Invalid cached sentiment labels; remove data/sentiment.json and rerun")\n    if any(isinstance(v, bool) or not isinstance(v, (int, float)) for v in values.values()):\n        raise ValueError("Invalid cached sentiment type")\n    probs = np.array(list(values.values()), dtype=float)\n    if not np.isfinite(probs).all() or (probs < 0).any() or (probs > 1).any() or not np.isclose(probs.sum(), 1.0, atol=1e-5):\n        raise ValueError("Invalid cached sentiment probabilities; remove data/sentiment.json and rerun")\n\n\ndef score_posts(posts, batch_size=16):\n    """Frozen safe-weights inference; checkpoints are data, never Python pickle."""\n    import torch\n    from transformers import AutoTokenizer, AutoModelForSequenceClassification\n    path = ROOT / "data" / "sentiment.json"\n    signature = hashlib.sha256((MODEL_REV + "|max_length=128|" + "|".join(posts.tweet_id.astype(str) + posts.text)).encode()).hexdigest()\n    cache = json.loads(path.read_text()) if path.exists() else {"signature": signature, "scores": {}}\n    if cache.get("signature") != signature:\n        cache = {"signature": signature, "scores": {}}\n    remaining = posts[~posts.tweet_id.astype(str).isin(cache["scores"])]\n    if not remaining.empty:\n        os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")\n        torch.set_num_threads(4)\n        torch.manual_seed(42)\n        tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=MODEL_REV, trust_remote_code=False, token=False)\n        model = AutoModelForSequenceClassification.from_pretrained(\n            MODEL, revision=MODEL_REV, trust_remote_code=False, use_safetensors=True, token=False,\n        ).eval()\n        labels = {str(v).upper(): int(k) for k, v in model.config.id2label.items()}\n        if set(labels) != {"NEUTRAL", "BULLISH", "BEARISH"}:\n            raise ValueError(f"Unexpected sentiment label mapping: {labels}")\n        for start in range(0, len(remaining), batch_size):\n            batch = remaining.iloc[start:start + batch_size]\n            inputs = tokenizer(batch.text.tolist(), padding=True, truncation=True, max_length=128, return_tensors="pt")\n            with torch.inference_mode():\n                probs = model(**inputs).logits.softmax(-1).cpu().numpy()\n            if not np.isfinite(probs).all() or not np.allclose(probs.sum(axis=1), 1, atol=1e-5):\n                raise ValueError("Invalid sentiment probabilities")\n            for tid, values in zip(batch.tweet_id.astype(str), probs):\n                cache["scores"][tid] = {key.lower(): float(values[idx]) for key, idx in labels.items()}\n            atomic_json(path, cache)  # resumable after every batch\n            print(f"Sentiment: {min(start + batch_size, len(remaining))}/{len(remaining)} uncached posts", flush=True)\n    records = []\n    for row in posts.to_dict("records"):\n        values = cache["scores"][str(row["tweet_id"])]\n        validate_sentiment_scores(values)\n        row.update(values)\n        row["sentiment"] = values["bullish"] - values["bearish"]\n        records.append(row)\n    if not records:\n        raise ValueError("No eligible posts; cannot claim a sentiment experiment")\n    return pd.DataFrame(records)\n\n\ndef _read_pinned(name, sha256):\n    """Local pinned copies only. A missing or edited file fails closed."""\n    path = ROOT / "data" / name\n    if not path.exists():\n        return None\n    if digest(path) != sha256:\n        raise ValueError(f"{name} SHA-256 does not match the pinned copy")\n    return path\n\n\ndef load_hourly(coin):\n    """Hourly Binance spot bars, aggregated to the completed day before each cutoff."""\n    if coin not in COINS:\n        raise ValueError("Unknown coin")\n    path = _read_pinned(HOURLY_FILE, HOURLY_SHA256)\n    if path is None:\n        raise ValueError(f"data/{HOURLY_FILE} is required; see README for the pinned source")\n    columns = ["coin", "open_time", "open_usdt", "high_usdt", "low_usdt", "close_usdt",\n               "volume_usdt", "taker_buy_volume_usdt"]\n    bars = pd.read_csv(path, usecols=columns)\n    bars = bars[bars.coin.eq(coin)].copy()\n    bars["open_time"] = pd.to_datetime(bars.open_time, utc=True, errors="coerce")\n    bars = bars.sort_values("open_time").reset_index(drop=True)\n    if bars.empty or bars.open_time.isna().any() or bars.open_time.duplicated().any():\n        raise ValueError(f"Invalid hourly index for {coin}")\n    if not bars.open_time.dt.minute.eq(0).all():\n        raise ValueError("Hourly bars must open on the hour")\n    if not (bars.open_time.diff().dropna() == pd.Timedelta(hours=1)).all():\n        raise ValueError(f"Hourly bars are not contiguous for {coin}; no silent interpolation")\n    values = bars[["open_usdt", "high_usdt", "low_usdt", "close_usdt", "volume_usdt",\n                   "taker_buy_volume_usdt"]].to_numpy(float)\n    if not np.isfinite(values).all() or (values[:, :4] <= 0).any() or (values[:, 4:] < 0).any():\n        raise ValueError("Non-finite or invalid hourly market data")\n    if (bars.high_usdt < bars[["open_usdt", "close_usdt", "low_usdt"]].max(axis=1)).any():\n        raise ValueError("Invalid hourly OHLC bounds")\n    if (bars.low_usdt > bars[["open_usdt", "close_usdt", "high_usdt"]].min(axis=1)).any():\n        raise ValueError("Invalid hourly OHLC bounds")\n    bars["day"] = bars.open_time.dt.floor("D")\n    bars["hour_ret"] = np.log(bars.close_usdt / bars.open_usdt)\n    grouped = bars.groupby("day")\n    daily = pd.DataFrame({\n        "hourly_vol1": grouped.hour_ret.std(ddof=1),\n        "hourly_range1": (grouped.high_usdt.max() - grouped.low_usdt.min()) / grouped.close_usdt.last(),\n        "last_hour_ret": grouped.hour_ret.last(),\n        "taker_ratio1": grouped.taker_buy_volume_usdt.sum() / grouped.volume_usdt.sum(),\n        "hours": grouped.size(),\n    })\n    daily = daily[daily.hours.eq(24)].drop(columns="hours")\n    if daily.empty:\n        raise ValueError(f"No complete UTC days of hourly bars for {coin}")\n    # A day\'s 23:00 bar closes at its own midnight, so it informs the NEXT cutoff.\n    daily.index = daily.index + pd.Timedelta(days=1)\n    return daily.reset_index(names="date")\n\n\ndef load_btc():\n    """BTC/USDT daily context from the pinned independent market file."""\n    path = _read_pinned(EXPANDED_FILE, EXPANDED_SHA256)\n    if path is None:\n        raise ValueError(f"data/{EXPANDED_FILE} is required; see README for the pinned source")\n    frame = pd.read_csv(path, usecols=["utc_day", "coin", "open_usdt", "close_usdt", "btc_close_usdt"])\n    btc = frame[["utc_day", "btc_close_usdt"]].dropna().drop_duplicates("utc_day").copy()\n    btc["date"] = pd.to_datetime(btc.utc_day, utc=True, errors="coerce")\n    btc = btc.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)\n    if btc.empty or btc.date.duplicated().any():\n        raise ValueError("Invalid BTC date index")\n    if not (btc.date.diff().dropna() == pd.Timedelta(days=1)).all():\n        raise ValueError("BTC daily series has gaps; no silent forward filling")\n    closes = btc.btc_close_usdt.to_numpy(float)\n    if not np.isfinite(closes).all() or (closes <= 0).any():\n        raise ValueError("Invalid BTC closes")\n    btc["btc_ret1"] = btc.btc_close_usdt.pct_change().shift(1)\n    btc["btc_ret7"] = btc.btc_close_usdt.pct_change(7).shift(1)\n    # Same-provider cross-check: the independent daily file must reproduce our own bars exactly.\n    for coin in COINS:\n        local = ROOT / "data" / f"{coin}-prices.csv"\n        if not local.exists():\n            continue\n        mine = pd.read_csv(local)\n        mine["date"] = pd.to_datetime(mine.date, utc=True)\n        theirs = frame[frame.coin.eq(coin)].copy()\n        theirs["date"] = pd.to_datetime(theirs.utc_day, utc=True, errors="coerce")\n        merged = mine.merge(theirs[["date", "open_usdt", "close_usdt"]], on="date", how="inner")\n        if merged.empty:\n            raise ValueError(f"No overlapping daily bars to cross-check for {coin}")\n        if not np.array_equal(merged.open.to_numpy(float), merged.open_usdt.to_numpy(float)):\n            raise ValueError(f"Independent daily opens disagree with our Binance bars for {coin}")\n        if not np.array_equal(merged.close.to_numpy(float), merged.close_usdt.to_numpy(float)):\n            raise ValueError(f"Independent daily closes disagree with our Binance bars for {coin}")\n    return btc[["date", "btc_ret1", "btc_ret7"]]\n\n\ndef independent_check(scored):\n    """Agreement with a separately produced FinTwitBERT pass on a different pinned revision."""\n    path = _read_pinned(INDEPENDENT_FILE, INDEPENDENT_SHA256)\n    if path is None:\n        return {"available": False, "reason": f"data/{INDEPENDENT_FILE} is absent"}\n    other = pd.read_csv(path, usecols=["post_id", "coin", "p_bearish", "p_neutral", "p_bullish"])\n    other["post_id"] = other.post_id.astype(str)\n    other = other.drop_duplicates("post_id")\n    mine = scored[["tweet_id", "text", "bullish", "neutral", "bearish"]].copy()\n    mine["post_id"] = mine.tweet_id.astype(str)\n    merged = mine.merge(other, on="post_id", how="inner")\n    if merged.empty:\n        return {"available": False, "reason": "no shared post IDs"}\n    classes = ["bearish", "neutral", "bullish"]\n    ours = merged[classes].to_numpy(float).argmax(axis=1)\n    theirs = merged[["p_" + name for name in classes]].to_numpy(float).argmax(axis=1)\n    ours_net = merged.bullish.to_numpy(float) - merged.bearish.to_numpy(float)\n    theirs_net = merged.p_bullish.to_numpy(float) - merged.p_bearish.to_numpy(float)\n    delta = np.abs(merged.bullish.to_numpy(float) - merged.p_bullish.to_numpy(float))\n    length = merged.text.astype(str).str.len().to_numpy(float)\n    short, long = length <= 400, length > 400\n    return {"available": True, "our_model_revision": MODEL_REV,\n            "independent_model_revision": INDEPENDENT_MODEL_REV,\n            "shared_posts": int(len(merged)),\n            "class_agreement": float((ours == theirs).mean()),\n            "net_sentiment_sign_agreement": float((np.sign(ours_net) == np.sign(theirs_net)).mean()),\n            "mean_abs_p_bullish_difference": float(delta.mean()),\n            "max_abs_p_bullish_difference": float(delta.max()),\n            "share_p_bullish_differing_over_0_25": float((delta > 0.25).mean()),\n            "length_diagnostic": {\n                "mean_abs_difference_short_text": float(delta[short].mean()) if short.any() else None,\n                "mean_abs_difference_long_text": float(delta[long].mean()) if long.any() else None,\n                "note": "Disagreement is flat across text-length buckets, so truncation does not explain it.",\n            },\n            "note": "Independent scoring pass from a different pinned checkpoint revision. This is a reproducibility check, not a ground-truth accuracy measure."}\n\n\ndef market_features(prices):\n    out = prices.copy()\n    returns = out.close.pct_change()\n    out["ret1"] = returns.shift(1)\n    out["ret3"] = out.close.pct_change(3).shift(1)\n    out["ret7"] = out.close.pct_change(7).shift(1)\n    out["vol7"] = returns.rolling(7).std().shift(1)\n    out["range1"] = ((out.high - out.low) / out.close).shift(1)\n    out["volume_change"] = np.log1p(out.volume).diff().shift(1)\n    out["actual_return"] = out.close / out.open - 1\n    out["actual_up"] = (out.actual_return > 0).astype(int)\n    return out\n\n\ndef build_dataset(posts, prices, hourly, btc):\n    frames = []\n    examples = {}\n    for coin in COINS:\n        subset = posts[posts.tickers.apply(lambda xs: coin in xs)]\n        group = subset.groupby("date")\n        daily = group.agg(post_count=("tweet_id", "size"), sentiment=("sentiment", "mean"),\n                          disagreement=("sentiment", "std"), bullish=("bullish", "mean"), bearish=("bearish", "mean"))\n        daily["disagreement"] = daily.disagreement.fillna(0)\n        frame = (market_features(prices[coin]).set_index("date").join(daily)\n                 .join(hourly[coin].set_index("date")).join(btc.set_index("date")).reset_index())\n        frame["post_count"] = frame.post_count.fillna(0).astype(int)\n        frame["has_posts"] = (frame.post_count > 0).astype(int)\n        # Zero is an imputation only; has_posts distinguishes unobserved text from neutral.\n        frame[SOCIAL] = frame[SOCIAL].fillna(0)\n        frame["coin"] = coin\n        frame = frame[(frame.date >= START) & (frame.date < END)].dropna(subset=MARKET)\n        # The primary row set stays exactly as published; extended inputs must be complete.\n        if not np.isfinite(frame[MARKET_EXT].to_numpy(float)).all():\n            raise ValueError(f"Extended market features are incomplete for {coin}; refusing to impute")\n        for date, samples in group:\n            examples[(coin, date.strftime("%Y-%m-%d"))] = [{\n                "text": r["text"][:700], "url": r["url"] if isinstance(r.get("url"), str) and r["url"].startswith(("https://twitter.com/", "https://x.com/")) else "",\n                "created_at": r["created_at"].isoformat(), "available_at": r["available_at"].isoformat(),\n            } for r in samples.sort_values("available_at").tail(3).to_dict("records")]\n        frames.append(frame)\n    dataset = pd.concat(frames).sort_values(["date", "coin"]).reset_index(drop=True)\n    if dataset.duplicated(["date", "coin"]).any() or not np.isfinite(dataset[MARKET_EXT + SOCIAL].to_numpy()).all():\n        raise ValueError("Invalid feature table")\n    return dataset, examples\n\n\ndef split_dates(frame):\n    dates = sorted(frame.date.unique())\n    if len(dates) < 150:\n        raise ValueError("Too few unique dates for the planned evaluation")\n    val_start = pd.Timestamp(dates[int(len(dates) * 0.6)])\n    test_start = pd.Timestamp(dates[int(len(dates) * 0.8)])\n    # Exclude the adjacent day so a full target window ends BEFORE next partition.\n    train = frame[frame.date + pd.Timedelta(days=1) < val_start].copy()\n    val = frame[(frame.date >= val_start) & (frame.date + pd.Timedelta(days=1) < test_start)].copy()\n    test = frame[frame.date >= test_start].copy()\n    for part in (train, val, test):\n        if part.actual_up.nunique() != 2 or set(part.coin) != set(COINS):\n            raise ValueError("Each partition must cover five coins and both outcomes")\n    return train, val, test\n\n\ndef classifier(features, c):\n    prep = ColumnTransformer([\n        ("numeric", StandardScaler(), features),\n        ("coin", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["coin"]),\n    ])\n    return Pipeline([("prep", prep), ("model", LogisticRegression(C=c, max_iter=1000, random_state=42))])\n\n\ndef scores(y, prob):\n    prob = np.asarray(prob)\n    if len(y) == 0:\n        return {"accuracy": None, "balanced_accuracy": None, "brier": None, "n": 0}\n    if not np.isfinite(prob).all() or (prob < 0).any() or (prob > 1).any():\n        raise ValueError("Invalid forecast probabilities")\n    return {"accuracy": float(accuracy_score(y, prob >= 0.5)),\n            "balanced_accuracy": float(balanced_accuracy_score(y, prob >= 0.5)),\n            "brier": float(brier_score_loss(y, prob)), "n": int(len(y))}\n\n\ndef metric(name, frame, prob):\n    out = {"name": name, **scores(frame.actual_up, prob), "n_dates": int(frame.date.nunique()), "per_coin": {}}\n    for coin in COINS:\n        mask = frame.coin.eq(coin).to_numpy()\n        out["per_coin"][coin] = scores(frame.loc[mask, "actual_up"], np.asarray(prob)[mask])\n    return out\n\n\ndef bootstrap_delta(test, market, social):\n    # ponytail: 7-day moving blocks, exploratory interval; not a formal edge test.\n    tmp = pd.DataFrame({"date": test.date, "delta": (social - test.actual_up.to_numpy()) ** 2 - (market - test.actual_up.to_numpy()) ** 2})\n    series = tmp.groupby("date").delta.mean().to_numpy()\n    rng = np.random.default_rng(42)\n    draws = []\n    for _ in range(1000):\n        starts = rng.integers(0, len(series), size=int(np.ceil(len(series) / 7)))\n        sample = np.concatenate([series[(start + np.arange(7)) % len(series)] for start in starts])[:len(series)]\n        draws.append(float(sample.mean()))\n    return {"mean_social_minus_market_brier": float(series.mean()), "interval95": [float(x) for x in np.quantile(draws, [0.025, 0.975])],\n            "method": "1000 circular moving-block resamples of UTC dates, 7-day blocks; coins grouped by day; exploratory interval"}\n\n\ndef fit_and_report(raw, posts, prices, hourly, btc):\n    frame, examples = build_dataset(posts, prices, hourly, btc)\n    train, val, test = split_dates(frame)\n    configurations = {"Market only": MARKET, "Market + attention": MARKET + ATTENTION, "Market + X": MARKET + SOCIAL}\n    models, validation = {}, []\n    # Freeze feature definitions, candidates and threshold before viewing final test.\n    for name, features in configurations.items():\n        candidates = []\n        for c in (0.01, 0.1, 1.0):\n            fitted = classifier(features, c).fit(train, train.actual_up)\n            brier = brier_score_loss(val.actual_up, fitted.predict_proba(val)[:, 1])\n            candidates.append((float(brier), c, fitted))\n        best_brier, best_c, selected = min(candidates, key=lambda x: (x[0], x[1]))\n        models[name] = selected  # no refit on validation; final models see train only\n        validation.append({"name": name, "selected_C": best_c, "brier": best_brier, "candidates": [{"C": c, "brier": score} for score, c, _ in candidates]})\n    probabilities = {name: fitted.predict_proba(test)[:, 1] for name, fitted in models.items()}\n    # Declared secondary comparison: identical rows, identical protocol, richer market inputs.\n    extended_configurations = {"Market + BTC + intraday": MARKET_EXT,\n                               "Market + BTC + intraday + X": MARKET_EXT + SOCIAL}\n    ext_models, ext_validation = {}, []\n    for name, features in extended_configurations.items():\n        candidates = []\n        for c in (0.01, 0.1, 1.0):\n            fitted = classifier(features, c).fit(train, train.actual_up)\n            brier = brier_score_loss(val.actual_up, fitted.predict_proba(val)[:, 1])\n            candidates.append((float(brier), c, fitted))\n        best_brier, best_c, selected = min(candidates, key=lambda x: (x[0], x[1]))\n        ext_models[name] = selected\n        ext_validation.append({"name": name, "selected_C": best_c, "brier": best_brier,\n                               "candidates": [{"C": c, "brier": score} for score, c, _ in candidates]})\n    ext_probabilities = {name: fitted.predict_proba(test)[:, 1] for name, fitted in ext_models.items()}\n    EXT_BASE = "Market + BTC + intraday"\n    EXT_X = "Market + BTC + intraday + X"\n    majority = int(train.actual_up.mean() >= 0.5)\n    metric_rows = [metric("Always up", test, np.ones(len(test))),\n                   metric("Training majority", test, np.full(len(test), majority, dtype=float)),\n                   metric("Previous direction", test, (test.ret1.to_numpy() > 0).astype(float)),\n                   metric("Training prevalence", test, np.full(len(test), float(train.actual_up.mean())))]\n    metric_rows += [metric(name, test, p) for name, p in probabilities.items()]\n    predictions = []\n    for idx, row in enumerate(test.to_dict("records")):\n        date = row["date"].strftime("%Y-%m-%d")\n        prediction = {key: row[key] for key in ["coin", "actual_return", "actual_up", "post_count"]}\n        prediction.update(date=date, cutoff=row["date"].isoformat(), target_end=(row["date"] + pd.Timedelta(days=1)).isoformat(),\n                          market_prob=float(probabilities["Market only"][idx]), social_prob=float(probabilities["Market + X"][idx]),\n                          attention_prob=float(probabilities["Market + attention"][idx]),\n                          extended_market_prob=float(ext_probabilities[EXT_BASE][idx]),\n                          extended_social_prob=float(ext_probabilities[EXT_X][idx]), persistence=int(row["ret1"] > 0),\n                          examples=examples.get((row["coin"], date), []))\n        prediction.update({key: float(row[key]) if row["post_count"] else None for key in ["sentiment", "bullish", "bearish"]})\n        predictions.append(prediction)\n    coverage = []\n    for coin in COINS:\n        source_mask = raw.tickers.apply(lambda x: coin in x if isinstance(x, (list, np.ndarray)) else False)\n        selected = posts[posts.tickers.apply(lambda xs: coin in xs)]\n        coverage.append({"coin": coin, "total_posts": int(source_mask.sum()), "original_posts": int((source_mask & raw.tweet_type.eq("tweet")).sum()),\n                         "eligible_posts": int(len(selected)), "days_with_posts": int(selected.date.nunique()),\n                         "price_days": int(len(prices[coin])), "first": str(prices[coin].date.min().date()), "last": str(prices[coin].date.max().date())})\n    learning_curve = []\n    for fraction in (0.25, 0.5, 1.0):\n        dates = sorted(train.date.unique())\n        small = train[train.date.isin(dates[-max(30, int(len(dates) * fraction)):])]\n        c = next(x["selected_C"] for x in validation if x["name"] == "Market + X")\n        fitted = classifier(MARKET + SOCIAL, c).fit(small, small.actual_up)\n        learning_curve.append({"history_fraction": fraction, "train_dates": int(small.date.nunique()), "validation_brier": float(brier_score_loss(val.actual_up, fitted.predict_proba(val)[:, 1]))})\n    mask = test.post_count.to_numpy() > 0\n    subgroup = {"with_observed_posts": {"n": int(mask.sum()), "metrics": [metric(name, test[mask], p[mask]) for name, p in probabilities.items()] if mask.any() else []},\n                "without_observed_posts": {"n": int((~mask).sum())}}\n    extended = {\n        "note": "Secondary comparison on the same held-out rows: does richer market context change the answer? Not used for selection.",\n        "features": {"BTC": BTC, "intraday": INTRADAY},\n        "validation": ext_validation,\n        "metrics": [metric("Market only (primary)", test, probabilities["Market only"]),\n                    metric("Market + X (primary)", test, probabilities["Market + X"])]\n                   + [metric(name, test, p) for name, p in ext_probabilities.items()],\n        "uncertainty": bootstrap_delta(test, ext_probabilities[EXT_BASE], ext_probabilities[EXT_X]),\n        "primary_uncertainty": bootstrap_delta(test, probabilities["Market only"], probabilities["Market + X"]),\n    }\n    report = {\n        "meta": {"project": "HypeCheck", "generated_at": datetime.now(timezone.utc).isoformat(),\n                 "dataset": DATASET, "dataset_revision": DATASET_REV, "sentiment_model": MODEL, "sentiment_revision": MODEL_REV,\n                 "disclaimer": "This project is a prototype and is not financial advice.",\n                 "splits": {name: [part.date.min().strftime("%Y-%m-%d"), part.date.max().strftime("%Y-%m-%d")] for name, part in [("train", train), ("validation", val), ("test", test)]},\n                 "evaluation_scope": "Retrospective frozen-model historical replay. No live forecasts, orders, or promised returns.",\n                 "retrospective_pretrained_cutoff": START.strftime("%Y-%m-%d"),\n                 "eligible_posts_missing_collection_timestamp": int(posts.first_posted_at.isna().sum()),\n                 "availability_policy": "Original posts only; max(created_at, first_posted_at) where available. Prior-day posts only.",\n                 "target": "Positive Binance USDT open-to-close return during the next UTC day; flat counts as not up.",\n                 "hourly_market_source": "Binance Vision 1h spot klines from the pinned independent data package, aggregated only over UTC hours that closed before the cutoff.",\n                 "btc_context_source": "BTC/USDT daily closes from the pinned independent market file, cross-checked bar-for-bar against our own Binance daily bars.",\n                 "independent_scoring_revision": INDEPENDENT_MODEL_REV,\n                 "limitations": [\n                     "Selected influencers are not representative of all X. Missing observations do not mean no discussion occurred.",\n                     "Sparse and changing coverage; author/source changes can mimic sentiment effects. No causal claim.",\n                     "Pretrained sentiment checkpoint was public before the experiment period, but training-corpus overlap is not fully auditable.",\n                     "English sentiment model; source language is not independently verified. Multi-coin posts share one sentiment score.",\n                     "Sentiment is truncated to 128 tokens. Probabilities are not independently calibrated; sentiment certainty is not return certainty.",\n                     "Historical USDT exchange bars exclude fees, slippage, latency and execution constraints; no profitability claim.",\n                     "Only one held-out period. Date-block intervals are exploratory, and the final period must not be reused to tune models.",\n                     "Two pinned revisions of the same sentiment checkpoint disagree on the direction of about 13% of shared posts, so these scores are model-revision dependent rather than a stable ground truth.",\n                 ]},\n        "metrics": metric_rows, "coverage": coverage, "predictions": predictions,\n        "validation": validation, "learning_curve": learning_curve, "subgroups": subgroup,\n        "label_counts": {name: {coin: {"n": int(len(part[part.coin.eq(coin)])), "up": int(part.loc[part.coin.eq(coin), "actual_up"].sum())} for coin in COINS} for name, part in [("train", train), ("validation", val), ("test", test)]},\n        "uncertainty": bootstrap_delta(test, probabilities["Market only"], probabilities["Market + X"]),\n        "extended": extended,\n        "independent_check": independent_check(posts),\n    }\n    atomic_json(ROOT / "docs" / "report.json", report)\n    frame.to_csv(ROOT / "data" / "features.csv", index=False)\n    exports = {}\n    for name, fitted in models.items():\n        names = fitted.named_steps["prep"].get_feature_names_out().tolist()\n        exports[name] = {"feature_names": names, "coefficients": fitted.named_steps["model"].coef_[0].tolist(),\n                         "intercept": fitted.named_steps["model"].intercept_.tolist(),\n                         "numeric_mean": fitted.named_steps["prep"].named_transformers_["numeric"].mean_.tolist(),\n                         "numeric_scale": fitted.named_steps["prep"].named_transformers_["numeric"].scale_.tolist()}\n    atomic_json(ROOT / "artifacts" / "coefficients.json", exports)\n    files = [p for p in sorted((ROOT / "data").glob("*")) if p.is_file() and p.suffix in {".csv", ".parquet", ".json"}]\n    files += [ROOT / "docs/report.json", ROOT / "artifacts/coefficients.json"]\n    manifest = {"dataset_revision": DATASET_REV, "model_revision": MODEL_REV, "seed": 42, "code_sha256": digest(Path(__file__)),\n                "files": {str(p.relative_to(ROOT)): digest(p) for p in files}}\n    atomic_json(ROOT / "artifacts" / "manifest.json", manifest)\n    for item in metric_rows:\n        print(f"{item[\'name\']}: accuracy={item[\'accuracy\']:.4f}, balanced={item[\'balanced_accuracy\']:.4f}, Brier={item[\'brier\']:.4f}, n={item[\'n\']}")\n    return report\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--stage", choices=["data", "sentiment", "train", "all"], default="all")\n    parser.add_argument("--batch-size", type=int, default=16)\n    args = parser.parse_args()\n    if not 1 <= args.batch_size <= 64:\n        parser.error("batch-size must be between 1 and 64")\n    raw, selected = load_posts()\n    print(f"Source posts: {len(raw)}; eligible original deduplicated posts: {len(selected)}", flush=True)\n    prices = {coin: load_prices(coin) for coin in COINS}\n    hourly = {coin: load_hourly(coin) for coin in COINS}\n    btc = load_btc()\n    print("Price coverage:", {coin: len(p) for coin, p in prices.items()}, flush=True)\n    print("Hourly coverage:", {coin: len(h) for coin, h in hourly.items()}, "BTC context rows:", len(btc), flush=True)\n    if args.stage == "data":\n        return\n    scored = score_posts(selected, args.batch_size)\n    if args.stage == "sentiment":\n        return\n    fit_and_report(raw, scored, prices, hourly, btc)\n\n\nif __name__ == "__main__":\n    main()\n', 'headline_study.py': '"""Does prior-day X-post sentiment predict next-day memecoin direction?\nFrozen dates, prior-only features, and text-kind sensitivity; no causal claim.\n"""\nimport argparse,json,re\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import brier_score_loss\nimport pipeline as core\nKINDS=(\'events\',\'promotion\',\'technical\',\'other\')\nPATTERNS={\n \'events\':r\'\\b(?:etf|approv(?:al|ed)|list(?:ing|ed)|delist|partnership|launch|upgrade|lawsuit|sec|regulat\\w*|hack(?:ed)?|exploit|burn(?:ed|ing)?|airdrop|adoption)\\b\',\n \'promotion\':r\'\\b(?:buy now|last chance|don.t miss|guaranteed|moon|fomo|ape in|send it|gem|100x|1000x|lambo)\\b|🚀\',\n \'technical\':r\'\\b(?:support|resistance|breakout|breakdown|trend(?:line)?|rsi|moving average|sma|ema|chart|consolidat\\w*|candle|volume|bullish|bearish|longs?|shorts?)\\b\',\n}\ndef headline_kind(text):\n for kind in KINDS[:-1]:\n  if re.search(PATTERNS[kind],str(text),re.I):return kind\n return \'other\'\ndef previous_day(prices):\n x=prices.sort_values(\'date\')\n result=(x.close>x.open).astype(float).shift(1)\n # The next row must actually be the next UTC day, not simply next available bar.\n result=result.where(x.date.diff().eq(pd.Timedelta(days=1)))\n return pd.Series(result.to_numpy(),index=x.date,name=\'previous_day\')\ndef aggregate_kinds(posts):\n records=[]\n for row in posts.to_dict(\'records\'):\n  target=pd.Timestamp(row[\'available_at\']).floor(\'D\')+pd.Timedelta(days=1)\n  for coin in set(row[\'tickers\'])&set(core.COINS):\n   records.append({\'coin\':coin,\'date\':target,\'kind\':headline_kind(row[\'text\']),\'sentiment\':float(row[\'sentiment\'])})\n r=pd.DataFrame(records);index=pd.MultiIndex.from_frame(r[[\'coin\',\'date\']].drop_duplicates())\n out=pd.DataFrame(index=index)\n for k in KINDS:\n  g=r[r.kind.eq(k)].groupby([\'coin\',\'date\']).sentiment.agg([\'size\',\'mean\'])\n  out[k+\'_count\']=g[\'size\'];out[k+\'_sentiment\']=g[\'mean\']\n return out.fillna(0)\ndef paired_interval(dates,delta,reps=1000):\n z=pd.DataFrame({\'date\':np.asarray(dates),\'delta\':np.asarray(delta,float)})\n groups=[g.delta.to_numpy() for _,g in z.groupby(\'date\',sort=True)]\n rng=np.random.default_rng(42);draws=[];n=len(groups)\n for _ in range(reps):\n  starts=rng.integers(0,n,size=int(np.ceil(n/7)))\n  ids=np.concatenate([(s+np.arange(7))%n for s in starts])[:n]\n  draws.append(float(np.concatenate([groups[i] for i in ids]).mean()))\n return {\'mean\':float(z.delta.mean()),\'interval95\':np.quantile(draws,[.025,.975]).tolist(),\n         \'method\':\'1000 paired circular seven-observed-date block resamples; coins kept together\'}\ndef holm(values):\n a=np.asarray(values,float);order=np.argsort(a);out=np.zeros(len(a));prev=0.\n for rank,i in enumerate(order):\n  prev=max(prev,min(1.,(len(a)-rank)*a[i]));out[i]=prev\n return out.tolist()\ndef block_sign_p(dates,delta,reps=2000):\n z=pd.DataFrame({\'date\':np.asarray(dates),\'delta\':np.asarray(delta,float)}).groupby(\'date\').delta.agg([\'sum\',\'size\'])\n sums=z[\'sum\'].to_numpy();blocks=np.array([sums[i:i+7].sum() for i in range(0,len(sums),7)])\n rng=np.random.default_rng(42);obs=abs(blocks.sum());null=abs((rng.choice([-1,1],size=(reps,len(blocks)))*blocks).sum(axis=1))\n return float((1+sum(null>=obs))/(reps+1))\ndef choose(train,val,features):\n candidates=[]\n for c in (.01,.1,1.):\n  m=core.classifier(features,c).fit(train,train.actual_up)\n  candidates.append((float(brier_score_loss(val.actual_up,m.predict_proba(val)[:,1])),c,m))\n loss,c,m=min(candidates,key=lambda x:(x[0],x[1]))\n return m,{\'C\':c,\'validation_brier\':loss}\ndef safe_metrics(y,p):return core.scores(y,p)\ndef main(universe=\'original\'):\n if universe==\'expanded\':\n  from expanded_inputs import load,COINS\n  raw,posts,prices,frame=load();core.COINS=COINS\n else:\n  raw,posts=core.load_posts();posts=core.score_posts(posts)\n  prices={c:core.load_prices(c) for c in core.COINS};hourly={c:core.load_hourly(c) for c in core.COINS};btc=core.load_btc()\n  frame,_=core.build_dataset(posts,prices,hourly,btc)\n frame=frame.merge(aggregate_kinds(posts).reset_index(),on=[\'coin\',\'date\'],how=\'left\',validate=\'one_to_one\')\n kindcols=[k+s for k in KINDS for s in (\'_count\',\'_sentiment\')];frame[kindcols]=frame[kindcols].fillna(0.)\n frame[\'previous_day\']=np.nan\n for c in core.COINS:\n  mask=frame.coin.eq(c);frame.loc[mask,\'previous_day\']=frame.loc[mask,\'date\'].map(previous_day(prices[c])).to_numpy()\n if frame.previous_day.isna().any():raise ValueError(\'Missing previous UTC-day direction.\')\n train,val,test=core.split_dates(frame)\n assert train.date.max()+pd.Timedelta(days=1)<val.date.min()\n assert val.date.max()+pd.Timedelta(days=1)<test.date.min()\n assert not set(train.date)&set(test.date)\n y=test.actual_up.to_numpy()\n configs={\'Sentiment + coin\':core.SOCIAL,\'Coin only\':[],\'Market only\':core.MARKET,\'Market + sentiment\':core.MARKET+core.SOCIAL}\n predictions={\'Always up\':np.ones(len(test)),\'Previous-day direction\':test.previous_day.to_numpy(),\n              \'Training majority\':np.full(len(test),float(train.actual_up.mean()>=.5)),\n              \'Training prevalence\':np.full(len(test),float(train.actual_up.mean()))}\n models={};selection={}\n for name,features in configs.items():\n  models[name],selection[name]=choose(train,val,features);predictions[name]=models[name].predict_proba(test)[:,1]\n primary=\'Sentiment + coin\';metrics={k:core.metric(k,test,p) for k,p in predictions.items()}\n paired={}\n for base in (\'Always up\',\'Previous-day direction\',\'Coin only\',\'Market only\',\'Training majority\'):\n  delta=((predictions[primary]>=.5)==y).astype(float)-((predictions[base]>=.5)==y).astype(float)\n  paired[base]=paired_interval(test.date,delta)\n paired[\'sentiment_increment_brier\']=paired_interval(test.date,(predictions[\'Market + sentiment\']-y)**2-(predictions[\'Market only\']-y)**2)\n # Type comparisons and definitions are fixed in code before this run. Exploratory, since this corpus has been reused.\n kinds=[];basep=predictions[\'Market only\']\n for k in KINDS:\n  features=core.MARKET+[k+\'_count\',k+\'_sentiment\'];m,sel=choose(train,val,features);p=m.predict_proba(test)[:,1]\n  mask=test[k+\'_count\'].to_numpy()>0;delta=(p-y)**2-(basep-y)**2\n  row={\'kind\':k,\'selection\':sel,\'train_posts\':int(train[k+\'_count\'].sum()),\'test_posts\':int(test[k+\'_count\'].sum()),\n       \'test_coin_days_with_kind\':int(mask.sum()),\'all_test_rows\':safe_metrics(y,p),\n       \'kind_present_rows\':{name:safe_metrics(y[mask],pr[mask]) for name,pr in {\'kind_model\':p,\'market_only\':basep,\'always_up\':np.ones(len(test)),\'previous_day\':test.previous_day.to_numpy()}.items()},\n       \'paired_brier_vs_market\':paired_interval(test.date,delta),\'block_sign_p\':block_sign_p(test.date,delta),\n       \'scope\':\'Exploratory fixed-rule buckets; overlapping coin-day slices; not manually validated semantic labels.\'}\n  predictions[\'Kind: \'+k]=p;kinds.append(row)\n for row,p in zip(kinds,holm([r[\'block_sign_p\'] for r in kinds])):row[\'holm_adjusted_p\']=p\n # Both controls refit on training data; do not merely shuffle labels when scoring existing predictions.\n features=core.SOCIAL;null=[];placebo=[];rng=np.random.default_rng(42)\n for _ in range(20):\n  labels=rng.permutation(train.actual_up.to_numpy())\n  m=core.classifier(features,selection[primary][\'C\']).fit(train,labels)\n  null.append(safe_metrics(y,m.predict_proba(test)[:,1])[\'accuracy\'])\n  a,b=train.copy(),test.copy()\n  a[features]=a[features].to_numpy()[rng.permutation(len(a))];b[features]=b[features].to_numpy()[rng.permutation(len(b))]\n  m=core.classifier(features,selection[primary][\'C\']).fit(a,a.actual_up)\n  placebo.append(safe_metrics(y,m.predict_proba(b)[:,1])[\'accuracy\'])\n # Four expanding windows over the same final period; C frozen using earlier validation only.\n wf={name:np.full(len(test),np.nan) for name in (primary,\'Market only\',\'Market + sentiment\')};windows=[]\n testdays=sorted(test.date.unique())\n for chunk in np.array_split(np.array(testdays,dtype=object),4):\n  start=pd.Timestamp(chunk[0]);tr=frame[frame.date+pd.Timedelta(days=1)<start]\n  mask=test.date.isin(chunk).to_numpy();part=test[mask];win={\'start\':str(start.date()),\'end\':str(pd.Timestamp(chunk[-1]).date()),\'n\':len(part),\'metrics\':{}}\n  for name in wf:\n   m=core.classifier(configs[name],selection[name][\'C\']).fit(tr,tr.actual_up);p=m.predict_proba(part)[:,1];wf[name][mask]=p\n   win[\'metrics\'][name]=safe_metrics(part.actual_up,p)\n  win[\'metrics\'][\'Always up\']=safe_metrics(part.actual_up,np.ones(len(part)))\n  win[\'metrics\'][\'Previous-day direction\']=safe_metrics(part.actual_up,part.previous_day.to_numpy());windows.append(win)\n walk={\'note\':\'Secondary expanding-window re-estimation on the same final dates, not an independent replication.\',\n       \'windows\':windows,\'pooled\':{name:core.metric(name,test,p) for name,p in wf.items()}}\n cache=json.loads((core.ROOT/\'data\'/(\'expanded-sentiment.json\' if universe==\'expanded\' else \'sentiment.json\')).read_text())\n report={\'question\':\'Do prior-day X-post headline-style texts add information about next-UTC-day memecoin direction?\',\n  \'universe\':universe,\'tweet_sentiment_counts\':{str(k):int(v) for k,v in posts[[\'bullish\',\'bearish\',\'neutral\']].idxmax(axis=1).value_counts().items()},\'primary_model\':primary,\'target\':\'close > open on target UTC day; flat counts as not up\',\n  \'source_description\':\'Original X posts, not newspaper headlines. Archive subset, not complete coverage of X.\',\n  \'protocol\':{\'coins\':list(core.COINS),\'splits\':{name:{\'start\':str(d.date.min().date()),\'end\':str(d.date.max().date()),\'rows\':len(d),\'dates\':d.date.nunique()} for name,d in [(\'train\',train),(\'validation\',val),(\'test\',test)]},\n     \'C_candidates\':[.01,.1,1.],\'selection\':\'validation Brier only; threshold 0.5; no refit on validation for frozen model\',\n     \'availability\':\'max(created_at, first_posted_at); prior UTC-day posts only\',\n     \'previous_day_baseline\':\'Previous calendar UTC-day close > open, not previous close-to-close return.\',\n     \'test_reuse\':\'Exploratory extension: this test period was inspected in earlier research. Not a fresh confirmatory test.\'},\n  \'model_source\':{\'model\':core.MODEL,\'revision\':core.MODEL_REV,\'dataset\':core.DATASET,\'dataset_revision\':core.DATASET_REV,\'score_signature\':cache[\'signature\']},\n  \'leakage_checks\':{\'shared_train_test_dates\':len(set(train.date)&set(test.date)),\'train_target_end_before_validation\':True,\'validation_target_end_before_test\':True,\n     \'prior_only_post_windows\':bool((posts.available_at<posts.date).all()),\'all_models_same_test_rows\':True},\n  \'coverage\':{\'market_bar_rows\':sum(len(p) for p in prices.values()),\'all_feature_coin_days\':len(frame),\'eligible_original_posts\':len(posts),\'test_coin_days_with_posts\':int((test.post_count>0).sum()),\'test_coin_days_total\':len(test)},\n  \'selection\':selection,\'metrics\':metrics,\'paired_accuracy_differences\':paired,\'kinds\':kinds,\'taxonomy\':PATTERNS,\n  \'controls\':{\'runs\':20,\'real_accuracy\':metrics[primary][\'accuracy\'],\'label_permuted_accuracy\':null,\'feature_placebo_accuracy\':placebo,\n       \'label_permuted_max\':max(null),\'feature_placebo_max\':max(placebo)},\'walk_forward\':walk,\n  \'investor_interpretation\':{\'justifies\':[\'Treat bullish posts as opinions, not validated forecasts.\',\'Use this study to question claims of reliable sentiment-based direction prediction.\'],\n     \'does_not_justify\':[\'Buying, selling, shorting or leverage from these predictions.\',\'Claiming posts cause returns, or that all memecoin text prediction is impossible.\',\'Expected profits, live trading efficacy or universal conclusions about all X posts.\']},\n  \'limitations\':[\'Frozen pretrained sentiment encoder plus trained logistic regression; encoder is not fine-tuned in this experiment.\',\n      \'Rule-based text kinds are approximate and not manually validated; group slices may overlap.\',\n      \'Sparse archive coverage and source concentration limit generalisation.\',\n      \'Pretraining-corpus overlap and collection/edited-text history are not fully auditable.\',\n      \'No fees, slippage, trade execution or profitability evaluation.\'],\n  \'disclaimer\':\'This project is a prototype and is not financial advice.\'}\n core.atomic_json(core.ROOT/\'docs/headline-report.json\',report)\n pred=test[[\'date\',\'coin\',\'actual_up\',\'actual_return\',\'post_count\',\'previous_day\']].copy()\n for name,p in predictions.items():pred[name]=p\n pred.to_csv(core.ROOT/\'artifacts/headline-predictions.csv\',index=False)\n m=models[primary];prep=m.named_steps[\'prep\'];coef=m.named_steps[\'model\']\n # Safe JSON parameters, no pickle. The notebook trains afresh and has full inference object.\n core.atomic_json(core.ROOT/\'artifacts/headline-model.json\',{\'features\':core.SOCIAL,\'selection\':selection[primary],\n     \'coefficient\':coef.coef_.tolist(),\'intercept\':coef.intercept_.tolist(),\'feature_names\':prep.get_feature_names_out().tolist(),\n     \'numeric_mean\':prep.named_transformers_[\'numeric\'].mean_.tolist(),\'numeric_scale\':prep.named_transformers_[\'numeric\'].scale_.tolist()})\n print(\'TEST PERIOD\',report[\'protocol\'][\'splits\'][\'test\'])\n for name,row in metrics.items():print(f"{name:25s} accuracy {row[\'accuracy\']:.4%} Brier {row[\'brier\']:.5f} n={row[\'n\']}")\n print(\'KIND COMPARISONS (exploratory, full-panel Brier change; negative favours text kind)\')\n for r in kinds:print(r[\'kind\'],\'observed coin-days\',r[\'test_coin_days_with_kind\'],\'delta\',r[\'paired_brier_vs_market\'],\'Holm p\',r[\'holm_adjusted_p\'])\n print(\'WALK FORWARD\',json.dumps(walk[\'pooled\'],default=str))\n print(\'WROTE docs/headline-report.json, artifacts/headline-predictions.csv, artifacts/headline-model.json\')\nif __name__==\'__main__\':\n parser=argparse.ArgumentParser();parser.add_argument(\'--universe\',choices=[\'original\',\'expanded\'],default=\'expanded\');args=parser.parse_args();main(args.universe)\n', 'expanded_inputs.py': '"""Expanded, naturally sampled memecoin inputs. No sentiment quotas or synthetic data."""\nimport hashlib,json,shutil\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport pipeline as core\nROOT=Path(__file__).resolve().parent\nCOINS=(\'DOGE\',\'SHIB\',\'PEPE\',\'BONK\',\'FLOKI\',\'WIF\',\'TRUMP\',\'NEIRO\',\'PENGU\',\'PNUT\',\'TURBO\',\'BOME\',\'MEME\')\nPANEL_SHA=\'4ea2b737ee19daff97414a444ec3ea5313b797b78705e1b56dbf055bd5d47a56\'\n\ndef select_posts(raw):\n x=raw.copy();x[\'created_at\']=pd.to_datetime(x.created_at,utc=True,errors=\'coerce\');x[\'first_posted_at\']=pd.to_datetime(x.first_posted_at,utc=True,errors=\'coerce\')\n x[\'available_at\']=x[[\'created_at\',\'first_posted_at\']].max(axis=1)\n x=x[x.created_at.notna()&x.available_at.notna()&x.text.notna()&x.tweet_type.eq(\'tweet\')].copy()\n x[\'text\']=x.text.astype(str).str.strip();x=x[x.text.str.len().between(8,4000)]\n x=x.sort_values(\'available_at\').drop_duplicates(\'text\')\n x=x[(x.available_at>=core.START-pd.Timedelta(days=1))&(x.available_at<core.END)].copy()\n x[\'tickers\']=x.tickers.apply(lambda a:list(a) if isinstance(a,(list,np.ndarray)) else [])\n x=x[x.tickers.apply(lambda a:bool(set(a)&set(COINS)))].copy()\n x[\'date\']=x.available_at.dt.floor(\'D\')+pd.Timedelta(days=1)\n return x\n\ndef inspect_posts(raw,price_path=\'data/expanded-memecoin-daily.csv\'):\n selected=select_posts(raw)\n print(\'Eligible original posts:\',len(selected))\n print(\'Coins:\',\', \'.join(COINS))\n print(selected[[\'available_at\',\'text\']].head(5).to_string(index=False))\n print(\'Market bars:\',pd.read_csv(price_path).shape[0])\n print(\'Natural sentiment counts are reported after scoring in the next section.\')\n return selected\n\ndef load_prices(path=None):\n p=Path(path) if path is not None else ROOT/\'data/expanded-memecoin-daily.csv\'\n if hashlib.sha256(p.read_bytes()).hexdigest()!=PANEL_SHA:raise ValueError(\'Expanded price panel SHA-256 mismatch\')\n d=pd.read_csv(p);d[\'date\']=pd.to_datetime(d.date,utc=True)\n if set(d.coin)!=set(COINS):raise ValueError(\'Coin panel changed\')\n out={}\n for c in COINS:\n  x=d[d.coin.eq(c)].drop(columns=\'coin\').sort_values(\'date\').reset_index(drop=True)\n  if x.date.duplicated().any() or not x.date.diff().iloc[1:].eq(pd.Timedelta(days=1)).all():raise ValueError(\'Missing/duplicate calendar bar for \'+c)\n  a=x[[\'open\',\'high\',\'low\',\'close\',\'volume\']].to_numpy(float)\n  if not np.isfinite(a).all() or (a[:,:4]<=0).any() or (a[:,4]<0).any():raise ValueError(\'Invalid market values\')\n  if (x.high<x[[\'open\',\'close\',\'low\']].max(axis=1)).any() or (x.low>x[[\'open\',\'close\',\'high\']].min(axis=1)).any():raise ValueError(\'Invalid OHLC bounds\')\n  out[c]=x\n return out\n\ndef build_frame(posts,prices):\n parts=[]\n for c in COINS:\n  sub=posts[posts.tickers.apply(lambda a:c in a)]\n  daily=sub.groupby(\'date\').agg(post_count=(\'tweet_id\',\'size\'),sentiment=(\'sentiment\',\'mean\'),disagreement=(\'sentiment\',\'std\'),bullish=(\'bullish\',\'mean\'),bearish=(\'bearish\',\'mean\'))\n  f=core.market_features(prices[c]).set_index(\'date\').join(daily).reset_index()\n  f[\'post_count\']=f.post_count.fillna(0).astype(int);f[\'has_posts\']=(f.post_count>0).astype(int);f[core.SOCIAL]=f[core.SOCIAL].fillna(0)\n  f[\'coin\']=c;f=f[(f.date>=core.START)&(f.date<core.END)].dropna(subset=core.MARKET);parts.append(f)\n frame=pd.concat(parts).sort_values([\'date\',\'coin\']).reset_index(drop=True)\n if frame.duplicated([\'date\',\'coin\']).any() or not np.isfinite(frame[core.MARKET+core.SOCIAL].to_numpy(float)).all():raise ValueError(\'Invalid expanded feature frame\')\n return frame\n\ndef split_data(frame):\n days=sorted(frame.date.unique());a=pd.Timestamp(days[int(.6*len(days))]);b=pd.Timestamp(days[int(.8*len(days))])\n train=frame[frame.date+pd.Timedelta(days=1)<a].copy();val=frame[(frame.date>=a)&(frame.date+pd.Timedelta(days=1)<b)].copy();test=frame[frame.date>=b].copy()\n for x in [train,val,test]:\n  if set(x.coin)!=set(COINS) or x.actual_up.nunique()!=2:raise ValueError(\'Incomplete coin or outcome support\')\n return train,val,test\n\ndef load():\n raw,original=core.load_posts();selected=select_posts(raw)\n temp=ROOT/\'.cache/expanded-headlines\';(temp/\'data\').mkdir(parents=True,exist_ok=True)\n path=temp/\'data/sentiment.json\';canonical=ROOT/\'data/expanded-sentiment.json\'\n if canonical.exists():shutil.copy2(canonical,path)\n elif (ROOT/\'data/sentiment.json\').exists():\n  old=json.loads((ROOT/\'data/sentiment.json\').read_text())\n  expected=hashlib.sha256((core.MODEL_REV+\'|max_length=128|\'+\'|\'.join(original.tweet_id.astype(str)+original.text)).encode()).hexdigest()\n  if old.get(\'signature\')==expected:\n   # Same pinned original dataset, model revision and text. Reuse overlapping frozen scores only.\n   ids=set(selected.tweet_id.astype(str));scores={k:v for k,v in old[\'scores\'].items() if k in ids}\n   for v in scores.values():core.validate_sentiment_scores(v)\n   signature=hashlib.sha256((core.MODEL_REV+\'|max_length=128|\'+\'|\'.join(selected.tweet_id.astype(str)+selected.text)).encode()).hexdigest()\n   path.write_text(json.dumps({\'signature\':signature,\'scores\':scores}))\n oldroot=core.ROOT\n try:\n  core.ROOT=temp;scored=core.score_posts(selected)\n finally:core.ROOT=oldroot\n shutil.copy2(path,canonical)\n prices=load_prices();frame=build_frame(scored,prices)\n return raw,scored,prices,frame\n', 'test_headline_study.py': "import unittest\nimport numpy as np\nimport pandas as pd\nfrom headline_study import headline_kind, previous_day, aggregate_kinds, paired_interval, holm\nclass HeadlineTests(unittest.TestCase):\n def test_taxonomy(self):\n  self.assertEqual(headline_kind('ETF approved for exchange listing'),'events')\n  self.assertEqual(headline_kind('Buy now 100x to the moon'),'promotion')\n  self.assertEqual(headline_kind('Rejected at resistance, support below'),'technical')\n  self.assertEqual(headline_kind('Thinking about the weekend'),'other')\n def test_exclusive_and_case(self):\n  self.assertEqual(headline_kind('BREAKOUT after ETF approval'),'events')\n def test_previous_direction_is_open_close_not_close_close(self):\n  prices=pd.DataFrame({'date':pd.date_range('2025-01-01',periods=3,tz='UTC'),'open':[100,90,110],'close':[95,100,105]})\n  out=previous_day(prices)\n  self.assertTrue(pd.isna(out.iloc[0]));self.assertEqual(out.iloc[1],0);self.assertEqual(out.iloc[2],1)\n def test_aggregation_uses_availability_day(self):\n  p=pd.DataFrame({'available_at':pd.to_datetime(['2025-01-01T23:59Z','2025-01-02T00:00Z']),\n    'text':['breakout','ETF approval'],'tickers':[['DOGE'],['DOGE']],'sentiment':[.5,-.5]})\n  g=aggregate_kinds(p)\n  self.assertEqual(g.loc[('DOGE',pd.Timestamp('2025-01-02',tz='UTC')),'technical_count'],1)\n  self.assertEqual(g.loc[('DOGE',pd.Timestamp('2025-01-03',tz='UTC')),'events_count'],1)\n def test_paired_identical(self):\n  d=pd.date_range('2025-01-01',periods=20,tz='UTC');x=np.ones(20)\n  r=paired_interval(d,x-x,reps=100);self.assertEqual(r['interval95'],[0.,0.])\n def test_holm(self):\n  q=holm([.01,.04,.03]);self.assertTrue(np.allclose(q,[.03,.06,.06]))\nif __name__=='__main__':unittest.main()\n", 'test_expanded_inputs.py': "import unittest,tempfile\nfrom pathlib import Path\nimport pandas as pd\nfrom expanded_inputs import select_posts,load_prices,split_data,COINS\nclass ExpandedTests(unittest.TestCase):\n def test_supported_original_posts_only(self):\n  r=pd.DataFrame({'tweet_id':['1','2','3'],'text':['$WIF resistance test','$WIF resistance test','$NOT supported example'],\n   'tweet_type':['tweet','retweet','tweet'],'created_at':['2025-01-01']*3,'first_posted_at':['2025-01-01']*3,'tickers':[['WIF'],['WIF'],['NOT']]})\n  x=select_posts(r);self.assertEqual(len(x),1);self.assertEqual(x.iloc[0].tweet_id,'1')\n def test_collection_delay_and_next_day(self):\n  r=pd.DataFrame({'tweet_id':['1'],'text':['$DOGE example post'],'tweet_type':['tweet'],'created_at':['2025-01-01T23:00Z'],'first_posted_at':['2025-01-02T01:00Z'],'tickers':[['DOGE']]})\n  x=select_posts(r).iloc[0];self.assertEqual(x.date,pd.Timestamp('2025-01-03',tz='UTC'));self.assertLess(x.available_at,x.date)\n def test_price_panel_tamper_rejected(self):\n  with tempfile.TemporaryDirectory() as t:\n   p=Path(t)/'bad.csv';p.write_text('bad data')\n   with self.assertRaises(ValueError):load_prices(p)\n def test_temporal_purges_and_support(self):\n  d=pd.DataFrame([{'date':x,'coin':c,'actual_up':i%2} for i,x in enumerate(pd.date_range('2024-03-01',periods=200,tz='UTC')) for c in COINS])\n  tr,v,te=split_data(d)\n  self.assertLess(tr.date.max()+pd.Timedelta(days=1),v.date.min());self.assertLess(v.date.max()+pd.Timedelta(days=1),te.date.min());self.assertEqual(set(te.coin),set(COINS))\n def test_inspection_does_not_require_sentiment_cache(self):\n  import os,io,contextlib\n  from expanded_inputs import inspect_posts\n  raw=pd.DataFrame({'tweet_id':['1'],'text':['DOGE example post'],'tweet_type':['tweet'],'created_at':['2025-01-01'],'first_posted_at':['2025-01-01'],'tickers':[['DOGE']]})\n  cwd=Path.cwd()\n  with tempfile.TemporaryDirectory() as t:\n   try:\n    os.chdir(t);Path('data').mkdir();pd.DataFrame({'close':[1]}).to_csv('data/expanded-memecoin-daily.csv',index=False)\n    with contextlib.redirect_stdout(io.StringIO()):result=inspect_posts(raw)\n    self.assertEqual(len(result),1)\n    self.assertFalse(Path('data/expanded-sentiment.json').exists())\n   finally:os.chdir(cwd)\nif __name__=='__main__':unittest.main()\n"}
for name,text in SOURCES.items():
    Path(name).write_text(text)
    print('Wrote',name)

Wrote pipeline.py
Wrote headline_study.py
Wrote expanded_inputs.py
Wrote test_headline_study.py
Wrote test_expanded_inputs.py


In [3]:
EMBEDDED_INPUTS = {'data/expanded-memecoin-daily.csv': {'sha256': '4ea2b737ee19daff97414a444ec3ea5313b797b78705e1b56dbf055bd5d47a56', 'zlib_base64': '...'}, 'data/expanded-sentiment.json': {'sha256': 'cee6a0649aa3484750e972593e27dd8d966639d5a8a6b1a1829ededcdce80ac2', 'zlib_base64': '...'}}
for name,item in EMBEDDED_INPUTS.items():
    payload=zlib.decompress(base64.b64decode(item['zlib_base64']))
    if hashlib.sha256(payload).hexdigest()!=item['sha256']:raise ValueError('Embedded input was changed: '+name)
    Path(name).write_bytes(payload)
    print('Verified',name,len(payload),'bytes')
import pipeline as core
from expanded_inputs import select_posts,load,split_data,COINS
RECOMPUTE_SENTIMENT=False
if RECOMPUTE_SENTIMENT:
    Path('data/expanded-sentiment.json').unlink(missing_ok=True)
    Path('.cache/expanded-headlines/data/sentiment.json').unlink(missing_ok=True)
raw,_=core.load_posts()  # pinned public source download + SHA verification
print('Verified original source rows:',len(raw))

Verified data/expanded-memecoin-daily.csv 906803 bytes
Verified data/expanded-sentiment.json 264043 bytes
Verified original source rows: 104379


## 2. Look at the data before training
Keep original posts, bounded nonempty text, exact supported ticker tags and earliest exact-text duplicates. The later of publication and archive collection time is conservative availability. A post from day d feeds only the forecast for day d+1.

Sentiment classes are outputs of the pretrained model, not human labels. Bullish does not mean the coin actually rose. Multi-coin posts can contribute to multiple coin-days; tweet count, market-bar count and prediction-row count are different units.

In [4]:
from expanded_inputs import inspect_posts
selected=inspect_posts(raw)
import pandas as pd,numpy as np

Eligible original posts: 1773
Coins: DOGE, SHIB, PEPE, BONK, FLOKI, WIF, TRUMP, NEIRO, PENGU, PNUT, TURBO, BOME, MEME
                 available_at                                                                                                                                                          text
2024-03-22 04:49:17.553+00:00                                                                                                                         Take partial profit fam 🔥 $bome #bome
2024-03-22 06:29:03.335+00:00                                                                                                              $doge legit best looking major by a mile atm imo
2024-03-22 07:19:03.469+00:00 The doge of this cycle should go for the next round anytime now IMO.\n\nAfter consolidation comes continuation.\nInvalidation below the horizontal.\n\n$FLOKI
2024-03-22 10:29:17.676+00:00                                               Bagged some \n\nBeing honest here. I think $doge gonna

## 3. Split into training, validation and later-date test data
**Never split these financial rows randomly.** Use shared chronological boundaries, approximately 60/20/20 across UTC dates, and purge the adjacent day at each boundary. Earlier target windows must end before the next partition begins.

The label is next-UTC-day **close > open**; flat counts as not up. Only strictly prior-day sentiment and lagged market features can be predictors. Scaling and coefficients are fitted on training rows; regularisation is selected by validation Brier error. The expanded universe and period were not a pristine new preregistration; we disclose archive reuse.

In [5]:
raw,scored,prices,frame=load()
print('Natural sentiment counts:',scored[['bullish','bearish','neutral']].idxmax(axis=1).value_counts().to_dict())
train,validation,test=split_data(frame)
print(pd.DataFrame([{'partition':name,'first':str(x.date.min().date()),'last':str(x.date.max().date()),'coin_days':len(x),'UTC_dates':x.date.nunique()} for name,x in [('train',train),('validation',validation),('test',test)]]).to_string(index=False))
assert train.date.max()+pd.Timedelta(days=1)<validation.date.min()
assert validation.date.max()+pd.Timedelta(days=1)<test.date.min()
assert not set(train.date)&set(test.date)
print('Boundary and shared-date leakage checks passed.')

Natural sentiment counts: {'bullish': 1238, 'bearish': 358, 'neutral': 177}
 partition      first       last  coin_days  UTC_dates
     train 2024-03-01 2025-09-29       6171        578
validation 2025-10-01 2026-04-10       2496        192
      test 2026-04-12 2026-10-21       2509        193
Boundary and shared-date leakage checks passed.


## 4. Baselines: always up and previous-day direction
Always up ignores all text. Persistence copies the previous calendar UTC day's **open-to-close** direction, matching the target—not previous close-to-close return. We also report training majority and a training-prevalence probability baseline so a falling market cannot masquerade as text skill.

In [6]:
from headline_study import previous_day
persistence=np.full(len(test),np.nan)
for coin in COINS:
    mask=test.coin.eq(coin).to_numpy()
    persistence[mask]=test.loc[mask,'date'].map(previous_day(prices[coin])).to_numpy()
assert np.isfinite(persistence).all()
y=test.actual_up.to_numpy()
baselines={'Always up':np.ones(len(test)),'Previous-day direction':persistence,'Training majority':np.full(len(test),float(train.actual_up.mean()>=.5)),'Training prevalence':np.full(len(test),train.actual_up.mean())}
print(pd.DataFrame([{'method':name,**core.scores(y,p)} for name,p in baselines.items()]).to_string(index=False))

                method  accuracy  balanced_accuracy    brier    n
             Always up  0.460741           0.500000 0.539259 2509
Previous-day direction  0.485054           0.481608 0.514946 2509
     Training majority  0.539259           0.500000 0.460741 2509
   Training prevalence  0.539259           0.500000 0.248602 2509


## 5. Model 1: pretrained sentiment + logistic regression
The primary model uses daily count/presence, mean bullish-minus-bearish sentiment, disagreement, bullish/bearish probabilities and coin identity. Frozen **FinTwitBERT** scores text; **logistic regression** learns the market-direction target. The encoder is not fine-tuned.

Fit C candidates 0.01, 0.1 and 1.0 on training only, choose using validation Brier, threshold at 0.5. Compare with coin-only to test whether sentiment adds anything beyond asset identity. All methods use the same later-date rows.

In [7]:
done=subprocess.run([sys.executable,'headline_study.py','--universe','expanded'],text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(done.stdout)
done.check_returncode()
report=json.loads(Path('docs/headline-report.json').read_text())
pred=pd.read_csv('artifacts/headline-predictions.csv')
print('CHECKS:',report['leakage_checks'])

TEST PERIOD {'start': '2026-04-12', 'end': '2026-10-21', 'rows': 2509, 'dates': 193}
Always up                 accuracy 46.0741% Brier 0.53926 n=2509
Previous-day direction    accuracy 48.5054% Brier 0.51495 n=2509
Training majority         accuracy 53.9259% Brier 0.46074 n=2509
Training prevalence       accuracy 53.9259% Brier 0.24860 n=2509
Sentiment + coin          accuracy 53.0889% Brier 0.24941 n=2509
Coin only                 accuracy 53.9259% Brier 0.24851 n=2509
Market only               accuracy 54.4041% Brier 0.24772 n=2509
Market + sentiment        accuracy 53.6070% Brier 0.24863 n=2509
KIND COMPARISONS (exploratory, full-panel Brier change; negative favours text kind)
events observed coin-days 16 delta {'mean': 0.0001305535788175643, 'interval95': [-0.0001789293487402769, 0.0005823252817242048], 'method': '1000 paired circular seven-observed-date block resamples; coins kept together'} Holm p 1.0
promotion observed coin-days 12 delta {'mean': -2.9249257720444738e-05, 'interv

## 6. Where Model 1 goes wrong
Show the most confident wrong forecasts, not only successes. A probability is not a guarantee; a mistaken market forecast is not evidence the underlying author was fraudulent. These examples are deliberately selected for error analysis, not used to adjust the trained model.

In [8]:
wrong=pred[(pred['Sentiment + coin']>=.5)!=pred.actual_up].copy()
wrong['confidence']=(wrong['Sentiment + coin']-.5).abs()
print(wrong.nlargest(5,'confidence')[['date','coin','actual_up','Sentiment + coin','post_count','actual_return']].to_string(index=False))
for row in wrong.nlargest(3,'confidence').to_dict('records'):
    day=pd.Timestamp(row['date'])
    relevant=scored[(scored.date==day)&scored.tickers.apply(lambda a:row['coin'] in a)]
    print('INPUT CONTEXT',row['coin'],str(day.date()),relevant.text.head(2).tolist())

                     date  coin  actual_up  Sentiment + coin  post_count  actual_return
2026-10-10 00:00:00+00:00  DOGE          1          0.367501           3       0.004450
2026-04-30 00:00:00+00:00  DOGE          1          0.370981           2       0.023350
2026-07-01 00:00:00+00:00 PENGU          1          0.386130           1       0.020958
2026-05-03 00:00:00+00:00 PENGU          1          0.389061           1       0.000506
2026-06-03 00:00:00+00:00   WIF          1          0.389519           1       0.085714
INPUT CONTEXT DOGE 2026-10-10 ['🐋 WHALE WATCH : Bitcoin slid 4.2% from $86592 to under $83000 triggering $550M in liquidations.\n\nMost altcoins took it harder. Seven day breakdown:\n\n=> $RENDER +4.4%\n=> $ADA 0.0%\n=> $BCH -3.6%\n=> $LTC -4.2%\n=> $SOL -4.6%\n=> $XRP -5.7%\n=> $DOGE -8.0%\n=> $LINK -9.6%\n=> $XLM -9.7%\n=> $ZEC -13.7%\n=> $UNI -16.3%\n\nBitcoin lost 1.5% on the week. Nine of eleven lost more. RENDER was the only green. ADA went flat. UNI and ZEC bot

## 7. Our addition: the larger later-date evaluation
No hand-selected 40 cases, no 15/25 sentiment quota. Evaluate the **entire natural test panel** and show results by coin, plus four expanding-date windows. Assets on the same date are dependent; the number of dates is not multiplied by the number of coins to claim independent evidence.

This is a larger version of the same archive, not an independent external dataset. Sparse text coverage remains a limitation. Missing archive posts do not prove the coin was not discussed.

In [9]:
print('TEST PANEL:',report['protocol']['splits']['test'])
print('TEXT COVERAGE:',report['coverage'])
print('NATURAL SENTIMENT COUNTS:',report['tweet_sentiment_counts'])
print(pd.DataFrame([{'coin':coin,**values} for coin,values in report['metrics']['Sentiment + coin']['per_coin'].items()]).to_string(index=False))
for window in report['walk_forward']['windows']:
    print(window['start'],window['end'],'n=',window['n'],{name:row['accuracy'] for name,row in window['metrics'].items()})
print('Expanding-window checks reuse these final dates; not independent replications.')

TEST PANEL: {'start': '2026-04-12', 'end': '2026-10-21', 'rows': 2509, 'dates': 193}
TEXT COVERAGE: {'market_bar_rows': 11432, 'all_feature_coin_days': 11202, 'eligible_original_posts': 1773, 'test_coin_days_with_posts': 219, 'test_coin_days_total': 2509}
NATURAL SENTIMENT COUNTS: {'bullish': 1238, 'bearish': 358, 'neutral': 177}
 coin  accuracy  balanced_accuracy    brier   n
 DOGE  0.461140           0.451022 0.253834 193
 SHIB  0.559585           0.524092 0.248184 193
 PEPE  0.538860           0.529282 0.249033 193
 BONK  0.580311           0.514063 0.247250 193
FLOKI  0.518135           0.493159 0.250757 193
  WIF  0.523316           0.492395 0.250137 193
TRUMP  0.544041           0.500920 0.249365 193
NEIRO  0.538860           0.496158 0.248755 193
PENGU  0.476684           0.449407 0.252805 193
 PNUT  0.606218           0.506494 0.242984 193
TURBO  0.544041           0.495283 0.247986 193
 BOME  0.487047           0.484536 0.251358 193
 MEME  0.523316           0.495691 0.249940 

## 8. One measured improvement: market context and kinds of posts
Compare sentiment alone, market history alone, and the combination. Then ask whether **events, promotion, technical/price action, or other** posts contribute differently.

Keyword buckets are exclusive per post in that priority order, approximate and not human-validated. Coin-day slices can overlap. Each kind's count and sentiment are added to market history, evaluated on all test rows. Report support, paired date-block intervals and Holm-adjusted block-sign tests. Negative Brier change favours adding the kind. A high score from a tiny slice is not a strategy.

In [10]:
from headline_study import PATTERNS
print('Fixed keyword rules:',json.dumps(PATTERNS,indent=2))
rows=[]
for item in report['kinds']:
    ci=item['paired_brier_vs_market']
    rows.append({'kind':item['kind'],'coin_days_with_kind':item['test_coin_days_with_kind'],'post_coin_incidences':item['test_posts'],'Brier_change':ci['mean'],'CI_low':ci['interval95'][0],'CI_high':ci['interval95'][1],'Holm_p':item['holm_adjusted_p']})
print(pd.DataFrame(rows).to_string(index=False))
print('Market + sentiment vs market-only paired Brier:',report['paired_accuracy_differences']['sentiment_increment_brier'])

Fixed keyword rules: {
  "events": "\\b(?:etf|approv(?:al|ed)|list(?:ing|ed)|delist|partnership|launch|upgrade|lawsuit|sec|regulat\\w*|hack(?:ed)?|exploit|burn(?:ed|ing)?|airdrop|adoption)\\b",
  "promotion": "\\b(?:buy now|last chance|don.t miss|guaranteed|moon|fomo|ape in|send it|gem|100x|1000x|lambo)\\b|\ud83d\ude80",
  "technical": "\\b(?:support|resistance|breakout|breakdown|trend(?:line)?|rsi|moving average|sma|ema|chart|consolidat\\w*|candle|volume|bullish|bearish|longs?|shorts?)\\b"
}
     kind  coin_days_with_kind  post_coin_incidences  Brier_change    CI_low  CI_high   Holm_p
   events                   16                    16      0.000131 -0.000179 0.000582 1.000000
promotion                   12                    13     -0.000029 -0.000326 0.000246 1.000000
technical                   53                    57      0.000124 -0.000052 0.000309 0.586207
    other                  161                   185      0.000587 -0.000003 0.001205 0.395802
Market + sentiment vs marke

## 9. The demo: inspect a historical forecast and reveal the market result
This uses the saved logistic model parameters, reconstructed from safe JSON—not a fabricated score. It is a historical forecast from prior information, not a live trade signal or a predictor of an individual tweet's causal effect. Change coin/date to another held-out row. Sentiment probabilities are not independently calibrated return probabilities.

In [11]:
saved=json.loads(Path('artifacts/headline-model.json').read_text())
def predict_saved(row):
    numeric=(row[saved['features']].to_numpy(dtype=float)-np.array(saved['numeric_mean']))/np.array(saved['numeric_scale'])
    categories=np.array([float(row['coin']==name.removeprefix('coin__coin_')) for name in saved['feature_names'][len(numeric):]])
    logit=float(np.r_[numeric,categories]@np.array(saved['coefficient'])[0]+saved['intercept'][0])
    return float(1/(1+np.exp(-logit)))
def replay(coin,date,reveal=False):
    day=pd.Timestamp(date,tz='UTC')
    row=test[(test.coin==coin)&(test.date==day)]
    if len(row)!=1:raise ValueError('Choose a supported coin and held-out UTC date.')
    row=row.iloc[0];prob=predict_saved(row)
    checked=pred[(pred.coin==coin)&(pd.to_datetime(pred.date,utc=True)==day)].iloc[0]
    assert np.isclose(prob,checked['Sentiment + coin'],atol=1e-12)
    context=scored[(scored.date==day)&scored.tickers.apply(lambda a:coin in a)]
    print(coin,date,'P(up), uncalibrated:',round(prob,4),'direction:','up' if prob>=.5 else 'not up')
    print('Eligible context:',context.text.head(3).tolist() or 'No eligible archive posts; not proof of silence.')
    if reveal:print('ACTUAL:',bool(row.actual_up),'return:',float(row.actual_return))
    else:print('Outcome hidden. Set reveal=True to inspect the real market result.')
first=pred[pred.post_count>0].sort_values(['date','coin']).iloc[0]
replay(first['coin'],str(first['date'])[:10],reveal=True)
print('Saved-model reconstruction verified against recorded prediction.')

PEPE 2026-04-21 P(up), uncalibrated: 0.5393 direction: up
Eligible context: ['The last final infinity stone to my bags $PEACE \n\nStrong narrative especially w/ the current war in the Middle East\n\n Matt Furie created PEACEFROG before $PEPE\n\nDouble narrative —> peace with the war & created before PEPE\n\nThis will be a core holding of mine moving forward\n✌️ 🐸']
ACTUAL: True return: 0.013297872340425565
Saved-model reconstruction verified against recorded prediction.


## 10. Results summary — model and baselines together
Use the full test-period comparison, not the most flattering coin, sentiment class or confidence band. Paired uncertainty groups coins by date. Both training-label permutation and feature-placebo controls actually refit the classifier; their accuracy nulls need not equal 50% under class imbalance.

In [12]:
order=['Always up','Previous-day direction','Training majority','Training prevalence','Sentiment + coin','Coin only','Market only','Market + sentiment']
print(pd.DataFrame([{'method':name,**{k:v for k,v in report['metrics'][name].items() if k in ['accuracy','balanced_accuracy','brier','n','n_dates']}} for name in order]).to_string(index=False))
print('Paired model-minus-baseline accuracy differences:',json.dumps(report['paired_accuracy_differences'],indent=2))
print('Controls:',report['controls'])
ci=report['paired_accuracy_differences']['Market only']['interval95']
print('Interpretation:', 'positive exploratory accuracy difference; needs future confirmation' if ci[0]>0 else ('model underperforms market-only here' if ci[1]<0 else 'no clear accuracy improvement over market-only demonstrated'))

                method  accuracy  balanced_accuracy    brier    n  n_dates
             Always up  0.460741           0.500000 0.539259 2509      193
Previous-day direction  0.485054           0.481608 0.514946 2509      193
     Training majority  0.539259           0.500000 0.460741 2509      193
   Training prevalence  0.539259           0.500000 0.248602 2509      193
      Sentiment + coin  0.530889           0.495955 0.249414 2509      193
             Coin only  0.539259           0.500000 0.248514 2509      193
           Market only  0.544041           0.513818 0.247719 2509      193
    Market + sentiment  0.536070           0.505923 0.248627 2509      193
Paired model-minus-baseline accuracy differences: {
  "Always up": {
    "mean": 0.07014746911119968,
    "interval95": [
      -0.016749701076125948,
      0.16023316062176163
    ],
    "method": "1000 paired circular seven-observed-date block resamples; coins kept together"
  },
  "Previous-day direction": {
    "mean": 

## 11. What this would and would not justify an investor doing
**Justified:** question claims that bullish posts reliably predict tomorrow; compare them with simple rules, uncertainty and actual outcomes.

**Not justified:** buying, selling, leverage, shorting, profit expectations, or saying posts caused returns. This result also does not prove all prediction impossible. No fees, slippage, execution or user-outcome study was performed.

The archive is sparse and selected, not a random sample of all X. Symbol collisions and retrospectively assigned ticker metadata may misattribute a post; this is not contract-address-verified pump.fun coverage. Current exchange listings create selection/survivorship limits. Later-listed coins have shorter training histories. Pretraining overlap, text edits and collection timing are not fully auditable. English sentiment truncates at 128 tokens. The period has been repeatedly explored; wider coverage is not independent confirmation.

*This project is a prototype and is not financial advice.*

## 12. Protocol tests and reproducibility

In [13]:
for filename in ['test_headline_study.py','test_expanded_inputs.py']:
    checked=subprocess.run([sys.executable,filename],text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
    print(checked.stdout);checked.check_returncode()
print('Protocol tests passed. Runtime evidence above distinguishes actual Colab from server execution.')

......
----------------------------------------------------------------------
Ran 6 tests in 0.040s

OK

.....
----------------------------------------------------------------------
Ran 5 tests in 0.056s

OK

Protocol tests passed. Runtime evidence above distinguishes actual Colab from server execution.


## Sources and credits
- Public original posts: https://huggingface.co/datasets/StephanAkkerman/financial-tweets-crypto ; dataset revision and SHA are in pipeline.py. Third-party post terms remain their own.
- Frozen sentiment: https://huggingface.co/StephanAkkerman/FinTwitBERT-sentiment ; pinned revision in pipeline.py, original work by Stephan Akkerman and Tim Koornstra.
- Actual Binance spot daily OHLCV bars, exact verified panel embedded above. Field definitions: https://github.com/binance/binance-spot-api-docs/blob/master/rest-api.md#klinecandlestick-data . Market data is not relicensed as our code.
- Original source and older independent hourly/BTC research remain credited in the repository; expanded primary models use daily bars, not invented hourly inputs.
- PyTorch, Transformers, scikit-learn, pandas, NumPy, PyArrow, Requests and dependencies.
- Hermes Agent / OpenAI coding assistance for research, implementation, review and testing. The team must understand and explain every part.
- Veritas AI official sample structure and Finance brief §4.2.4. We changed market/source and added time-aware evaluation, broader memecoins, text-kind checks, controls and a historical replay.